In [5]:
import numpy as np


def load_images(path):
    f = open(path, 'rb')         # open file in binary mode
    data = f.read()             #  read all data
    f.close()                     #  close file
    
    images = np.frombuffer(data, dtype=np.uint8, offset=16)    #skip header
    images = images.reshape(-1, 28*28)  #convert to (m, 784)
    
    return images / 255.0 # normalize (0–1)


def load_labels(path):
    f = open(path, 'rb')
    data = f.read()
    f.close()
    
    labels = np.frombuffer(data, dtype=np.uint8, offset=8)
    
    return labels


def one_hot(y):
    result = np.zeros((y.size, 10))
    
    for i in range(y.size):
        result[i][y[i]] = 1
    
    return result


def relu(x):
    return np.maximum(0, x)


def relu_derivative(x):
    return (x > 0).astype(float)


def softmax(x):
    exp = np.exp(x)
    return exp / np.sum(exp, axis=1, keepdims=True)


W1 = np.random.rand(784, 16)
b1 = np.zeros((1, 16))

W2 = np.random.rand(16, 16)
b2 = np.zeros((1, 16))

W3 = np.random.rand(16, 10)
b3 = np.zeros((1, 10))


def forward(X):               #X=(m*784)   w1=(784*16) b1=(1*16)
                                         #w2=(16*16) b2=(1*16)
                                         #w3=(16*10) b3=(1*10)
    Z1 = np.dot(X, W1) + b1  #(m*16)
    A1 = relu(Z1)            #(m*16)
    
    Z2 = np.dot(A1, W2) + b2 #(m*16)
    A2 = relu(Z2)            #(m*16)
    
    Z3 = np.dot(A2, W3) + b3 #(m*10)
    A3 = softmax(Z3)         #(m*10)
    
    return Z1, A1, Z2, A2, Z3, A3


def backward(X, Y, Z1, A1, Z2, A2, A3):
    m = X.shape[0]
    
    dZ3 = A3 - Y                   #(m*10)
    dW3 = np.dot(A2.T, dZ3) / m    #(16*10)=((16*m)*(m*10))
    db3 = np.sum(dZ3) / m          #(1*10)
    
    dZ2 = np.dot(dZ3, W3.T)        #(m*16)=((m*10)*(10*16))
    dZ2 = dZ2 * relu_derivative(Z2)
    dW2 = np.dot(A1.T, dZ2) / m    #(16*16)=((16*m)*(m*16))
    db2 = np.sum(dZ2) / m          #(1*16)
    
    dZ1 = np.dot(dZ2, W2.T)        #(m*16)=((m*16)*(16*16))
    dZ1 = dZ1 * relu_derivative(Z1)
    dW1 = np.dot(X.T, dZ1) / m     #(784*16)=((784*m)*(m*16))
    db1 = np.sum(dZ1) / m          #(1*16)
    
    return dW1, db1, dW2, db2, dW3, db3


def update(lr, dW1, db1, dW2, db2, dW3, db3):
    global W1, b1, W2, b2, W3, b3
    
    W1 -= lr * dW1
    b1 -= lr * db1
    
    W2 -= lr * dW2
    b2 -= lr * db2
    
    W3 -= lr * dW3
    b3 -= lr * db3


X = load_images(r"C:\ml\t10k-images.idx3-ubyte")
y = load_labels(r"C:\ml\t10k-labels.idx1-ubyte")

Y = one_hot(y)


for i in range(100):
    
    Z1, A1, Z2, A2, Z3, A3 = forward(X)
    
    dW1, db1, dW2, db2, dW3, db3 = backward(X, Y, Z1, A1, Z2, A2, A3)
    
    update(0.1, dW1, db1, dW2, db2, dW3, db3)
    
    if i % 10 == 0:
        predictions = np.argmax(A3, axis=1)
        accuracy = np.mean(predictions == y)
        print("Iteration:", i, "Accuracy:", accuracy)


predictions = np.argmax(A3, axis=1)

print("Final Accuracy:", np.mean(predictions == y))

C:\Users\Moksha\AppData\Local\Temp\ipykernel_34832\3783376588.py:43: RuntimeWarning: overflow encountered in exp
  exp = np.exp(x)
C:\Users\Moksha\AppData\Local\Temp\ipykernel_34832\3783376588.py:44: RuntimeWarning: invalid value encountered in divide
  return exp / np.sum(exp, axis=1, keepdims=True)


Iteration: 0 Accuracy: 0.0981
Iteration: 10 Accuracy: 0.098
Iteration: 20 Accuracy: 0.098
Iteration: 30 Accuracy: 0.098
Iteration: 40 Accuracy: 0.098
Iteration: 50 Accuracy: 0.098
Iteration: 60 Accuracy: 0.098
Iteration: 70 Accuracy: 0.098
Iteration: 80 Accuracy: 0.098
Iteration: 90 Accuracy: 0.098
Final Accuracy: 0.098
